# AI-powered Compliance Copilot: Auditing Software Requirements Against Engineering Standards

### Background

In safety-critical software engineering, such as NASA space missions, medical device development, and financial infrastructure every single software requirement must undergo a rigorous, clause-by-clause manual audit against strict compliance standards (e.g., NASA NPR 7150.2) before code can be written.

Currently, this compliance review relies on a small group of senior domain experts, creating a massive operational bottleneck and gatekeeping step. Automating this process transforms weeks of tedious manual review into rapid, structured first-pass reports that any engineer on the team can inspect and verify.

Therefore, we, the Aerospace 1A team, will build an AI-powered Compliance Copilot that reads software requirements, retrieves relevant clauses from an engineering compliance standard, evaluates whether each requirement satisfies the standard, and generates a structured audit report. The project follows a clear two-stage architectural evolution:

1. Non-Agentic Baseline: A deterministic Retrieval-Augmented Generation (RAG) pipeline that queries relevant standard clauses and judges requirement compliance.

2. Tool-Calling LLM Agent: A dynamic agent equipped with tools (`retrieve_clause`, `check_requirement`, `log_gap`) that orchestrates the audit workflow and compiles final JSON and Markdown gap reports.

## *September Milestones*

#### Importing the necessary packages, libraries, frameworks and modules

In [44]:
import requests
import os
import re
import json
import pandas as pd

!pip install pdfplumber -q
import pdfplumber

#### Loading the required datasets and documents (in the form of PDF files) into this notebook

In [45]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

url = "https://nodis3.gsfc.nasa.gov/npg_img/N_PR_7150_002D_/N_PR_7150_002D_.pdf"
response = requests.get(url, headers=headers)

with open('npr_full.pdf', 'wb') as f:
    f.write(response.content)

print(f"Downloaded npr_full.pdf: {len(response.content)} bytes")

Downloaded npr_full.pdf: 566622 bytes


#### Extracting all the requirements and rules clauses (starting from Section 3.1 all the way till Section 5.5 inclusive) from Chapter 3 till Chapter 5 inclusive, from the NASA PDF file

In [46]:
# Opening the NASA PDF file and extracting content from Chapters 3-5 inclusive
with pdfplumber.open('npr_full.pdf') as pdf:
    full_text = ""
    for page_no in range(19, 45):
        text = pdf.pages[page_no].extract_text()
        if text:
            full_text += text.strip() + "\n"

# Remove repeated page footer/boilerplate text
DEFAULT_BOILERPLATE_PATTERNS = [
    r"^NPR\s+[\w.\-]+\s*--.*$",
    r"^This document does not bind the public.*$",
    r"^incorporated into a contract\..*$",
    r"^the NASA Online Directives Information System.*$",
    r"^this is the correct version before use.*$",
    r"^Page \d+ of \d+.*$",
]

for boilerplate_pattern in DEFAULT_BOILERPLATE_PATTERNS:
    full_text = re.sub(boilerplate_pattern, "", full_text, flags=re.MULTILINE | re.IGNORECASE)

# Strip out standalone Chapter headings and 2-level titles (e.g. "Chapter 5: ...", "5.1 Software...")
full_text = re.sub(r'^\s*(?:Chapter\s+\d+.*|[3-5]\.\d+\s+[A-Za-z].*)$', '', full_text, flags=re.MULTILINE)

# Clean up duplicate clause headings caused by PDF page breaks
full_text = re.sub(r'(\d+\.\d+(?:\.\d+)?[\s\S]*?)\n+\s*\1', r'\1', full_text)

# Split by clause ID
ch3_5_raw = re.split(r'\n(?=\s*[3-5]\.\d+\.\d+)', full_text)

# Remove any leading and trailing whitespaces, tabs and new lines (data/clause cleaning)
ch3_5_clauses = [c.strip() for c in ch3_5_raw if c.strip()]

# Filter: Strictly retain and keep clauses following the clause ID patterns [3-5].x.x (no duplicates either)
ch3_5_clauses = [c for c in ch3_5_clauses if re.search(r'^\s*[3-5]\.\d+\.\d+', c)]

clauses = []
clauses.extend(ch3_5_clauses)

print(f"Found {len(clauses)} clauses\n")

for c in clauses:
  print(c[:100])
  print("---")

Found 113 clauses

3.1.1 Software life cycle planning covers the software aspects of a project from inception through
r
---
3.1.2 The project manager shall assess options for software acquisition versus development.
[SWE-033
---
3.1.3 The project manager shall develop, maintain, and execute software plans, including security
pl
---
3.1.4 The project manager shall track the actual results and performance of software activities
agai
---
3.1.5 The project manager shall define and document the acceptance criteria for the software.
[SWE-0
---
3.1.6 The project manager shall establish and maintain the software processes, software
documentatio
---
3.1.7 The project manager shall define and document the milestones at which the software
developer(s
---
3.1.8 The project manager shall require the software developer(s) to periodically report status and

---
3.1.9 The project manager shall require the software developer(s) to provide NASA with software
prod
---
3.1.10 The project manager shall req

#### Storing the extracted clauses (handoff for Data Triage)

In [47]:
chunks = []
no_swe_ids_clauses = []
for c in clauses:
    match = re.match(r'^\s*([3-5]\.\d+(?:\.\d+)?)\s+(.*)', c, re.DOTALL)
    if match:
        clause_id, text = match.groups()
        section_id = clause_id.rsplit('.', 1)[0]

        swe_match = re.search(r'\[(SWE-\d+)\]', text)
        if not swe_match:
          no_swe_ids_clauses.append(clause_id)
        else:
          # Separate main requirement text from any "Note:" paragraphs
          note_parts = re.split(r'\n(?=\s*Note(?:\s+\d+)?:)', text, flags=re.IGNORECASE)

          # The first part is the core requirement text
          req_text = note_parts[0].strip()

          # The remaining parts are notes attached to this requirement
          notes_list = [n.strip() for n in note_parts[1:] if n.strip()]

          chunks.append({
              "swe_id": swe_match.group(1),
              "section": section_id,
              "clause_id": clause_id,
              "text": req_text,
              "notes": notes_list
          })

with open('nasa_guidance_chunks.json', 'w') as f:
    json.dump(chunks, f, indent=2)

print(f"Saved {len(chunks)} clause chunks")
print()

chunks

Saved 99 clause chunks



[{'swe_id': 'SWE-033',
  'section': '3.1',
  'clause_id': '3.1.2',
  'text': 'The project manager shall assess options for software acquisition versus development.\n[SWE-033]',
  'notes': ['Note: The assessment can include risk, cost, and benefits criteria for each of the options\nlisted below:\na. Acquire an off-the-shelf software product that satisfies the requirement.\nb. Develop a software product or obtain the software service internally.\nc. Develop the software product or obtain the software service through contract.\nd. Enhance an existing software product or service.\ne. Reuse an existing software product or service.\nf. Source code available external to NASA.\nSee the NASA Software Engineering Handbook for additional detail.']},
 {'swe_id': 'SWE-013',
  'section': '3.1',
  'clause_id': '3.1.3',
  'text': 'The project manager shall develop, maintain, and execute software plans, including security\nplans, that cover the entire software life cycle and, as a minimum, address the 

Extracting all the other features/related important information of the software rules and requirements clauses from the table in Appendix C of the NASA PDF file; compiling and composing all features of the clauses into one big list of dictionaries, chunks, where each dictionary represents a row/clause

In [48]:
appendix_c_rows = []

with pdfplumber.open('npr_full.pdf') as pdf:
    for page_no in range(55, 80):
        page = pdf.pages[page_no]
        tables = page.extract_tables()
        for table in tables:
            for row in table:
                if len(row) > 2 and re.match(r'^\d{3}$', str(row[1]).strip()) and not re.match(r'^\d{3}$', str(row[2]).strip()):
                    row = [''] + row  # page 77 table is missing the leading column
                swe_num = str(row[2]).strip() if row[2] else ''
                # Skips headers ('SWE\n#') and empty strings ('')
                if not re.match(r'^\d{3}$', swe_num):
                    continue

                swe_id = f"SWE-{swe_num}"

                appendix_c_rows.append({
                    "swe_id": swe_id,
                    "raw_row": [c.strip() if c else "" for c in row]
                })

print(f"Extracted {len(appendix_c_rows)} requirement rows from Appendix C Matrix")


matrix_dict = {}

for item in appendix_c_rows:
  swe_id = item["swe_id"]
  row = item["raw_row"]

  section_num = row[1] if len(row) > 0 else None
  requirement_text = row[3] if len(row) > 1 else None
  authority = row[4] if len(row) > 2 else None

  class_marks = [col for col in row[5:10] if 'x' in col.lower()] if len(row) >= 8 else []

  f_auth = row[12] if len(row) > 8 else None
  f_mark = row[-1] if len(row) > 9 else None

  matrix_dict[swe_id] = {
      "matrix_section": section_num,
      "matrix_requirement_text": requirement_text.replace("\n", " ") if requirement_text else None,
      "matrix_authority": authority,
      "matrix_class_marks": len(class_marks),
      "matrix_f_authority": f_auth,
      "matrix_f_mark": f_mark
  }

for chunk in chunks:
  swe_id = chunk["swe_id"]
  chunk["sources"] = ["body"]

  if swe_id in matrix_dict:
    chunk["sources"].append("matrix")
    chunk.update(matrix_dict[swe_id])
  else:
    chunk.update({
            "matrix_section": None,
            "matrix_requirement_text": None,
            "matrix_authority": None,
            "matrix_class_marks": None,
            "matrix_f_authority": None,
            "matrix_f_mark": None
    })

with open('nasa_guidance_chunks_complete.json', 'w') as f:
  json.dump(chunks, f, indent=2)

print(f"Merged Appendix C matrix data into {len(chunks)} body requirement chunks!")

Extracted 100 requirement rows from Appendix C Matrix
Merged Appendix C matrix data into 99 body requirement chunks!


Converting the extracted software requirement clause chunks (currently each software requirement clause is stored in the form of a dictionary) to a Pandas Dataframe

In [49]:
reqs_clauses_df = pd.DataFrame(c for c in chunks)
reqs_clauses_df

,swe_id,section,clause_id,text,notes,sources,matrix_section,matrix_requirement_text,matrix_authority,matrix_class_marks,matrix_f_authority,matrix_f_mark
0,SWE-033,3.1,3.1.2,The project manager shall assess options for s...,"[Note: The assessment can include risk, cost, ...","[body, matrix]",3.1.2,The project manager shall assess options for s...,Center,5,X,
1,SWE-013,3.1,3.1.3,"The project manager shall develop, maintain, a...",[Note: The recommended practices and guideline...,"[body, matrix]",3.1.3,"The project manager shall develop, maintain, a...",Center,5,X,
2,SWE-024,3.1,3.1.4,The project manager shall track the actual res...,[],"[body, matrix]",3.1.4,The project manager shall track the actual res...,Center,4,X,
3,SWE-034,3.1,3.1.5,The project manager shall define and document ...,[],"[body, matrix]",3.1.5,The project manager shall define and document ...,Center,4,X,
4,SWE-036,3.1,3.1.6,The project manager shall establish and mainta...,[Note: A list of typical software engineering ...,"[body, matrix]",3.1.6,The project manager shall establish and mainta...,Center,4,X,
...,...,...,...,...,...,...,...,...,...,...,...,...
94,SWE-200,5.4,5.4.6,"The project manager shall collect, track, and ...",[],"[body, matrix]",5.4.6,"The project manager shall collect, track, and ...",Center,2,,
95,SWE-201,5.5,5.5.1,The project manager shall track and maintain s...,[],"[body, matrix]",5.5.1,The project manager shall track and maintain s...,Center,4,X,
96,SWE-202,5.5,5.5.2,The project manager shall define and implement...,"[Note: At a minimum, classes should include lo...","[body, matrix]",5.5.2,The project manager shall define and implement...,Center,3,X,
97,SWE-203,5.5,5.5.3,The project manager shall implement mandatory ...,"[Note: This includes operating systems, run-ti...","[body, matrix]",5.5.3,The project manager shall implement mandatory ...,Center,3,,


Generating a Schema for the Dataframe above (with the column names as the keys and their corresponding data types as values)

In [50]:
req_clauses_df_schema = reqs_clauses_df.dtypes.astype(str).to_dict()
req_clauses_df_schema

{'swe_id': 'object',
 'section': 'object',
 'clause_id': 'object',
 'text': 'object',
 'notes': 'object',
 'sources': 'object',
 'matrix_section': 'object',
 'matrix_requirement_text': 'object',
 'matrix_authority': 'object',
 'matrix_class_marks': 'int64',
 'matrix_f_authority': 'object',
 'matrix_f_mark': 'object'}

Adding classification tags or audit attributes to each requirement clause, thus placing each of them under a larger software requirements and rules category

In [51]:
def apply_category(clause_id):
  clauses_sections_to_categories = {
    "3.1": "Software Life Cycle Planning",
    "3.2": "Software Cost Estimation",
    "3.3": "Software Schedules",
    "3.4": "Software Training",
    "3.5": "Software Classification Assessments",
    "3.6": "Software Assurance and Software IV&V",
    "3.7": "Safety-Critical Software",
    "3.8": "Automatic Generation of Software Source Code",
    "3.9": "Software Development Processes and Practices",
    "3.10": "Software Reuse",
    "3.11": "Software Cybersecurity",
    "3.12": "Software Bi-Directional Traceability",
    "4.1": "Software Requirements",
    "4.2": "Software Architecture",
    "4.3": "Software Design",
    "4.4": "Software Implementation",
    "4.5": "Software Testing",
    "4.6": "Software Operations, Maintenance, and Retirement",
    "5.1": "Software Configuration Management",
    "5.2": "Software Risk Management",
    "5.3": "Software Peer Reviews/Inspections",
    "5.4": "Software Measurements",
    "5.5": "Software Non-Conformance or Defect Management"
  }

  # exact match on the first two parts (3.10.1 -> "3.10"), so 3.10 / 3.11 / 3.12 no longer fall into 3.1
  section_id = ".".join(str(clause_id).split(".")[:2])
  return clauses_sections_to_categories.get(section_id, "General Guidance")


# Adding functional category annotations based on clause numbers
reqs_clauses_df['functional_category'] = reqs_clauses_df['clause_id'].apply(apply_category)

Adding classification tags or audit attributes to each requirement clause, thus incorporating all the necessary action verbs that must be done in order to completely fulfill and pass the expectations of the clause

In [52]:
def extract_and_add_action_verbs(text):
  text_lower = str(text).lower()
  verbs = []
  for verb in ['develop', 'test', 'document', 'verify', 'track', 'maintain', 'assess', 'review']:
    if verb in text_lower:
      verbs.append(verb)

  return verbs

# Adding action verb annotations (what action(s) is/are required to completely, correctly, and properly fulfill each and every clause?)
reqs_clauses_df['action_verbs'] = reqs_clauses_df['text'].apply(extract_and_add_action_verbs)

# Saving annotated dataset
reqs_clauses_df.to_csv('annotated_nasa_clauses.csv', index=False)
print("Data annotation complete!")

Data annotation complete!


Taking a look at the first few rows of the new Dataframe that now includes two additional columns at the end of the table, `functional_category` and `action_verbs`, to see if those two columns actually exist and got integrated

In [53]:
reqs_clauses_df.head(20)

,swe_id,section,clause_id,text,notes,sources,matrix_section,matrix_requirement_text,matrix_authority,matrix_class_marks,matrix_f_authority,matrix_f_mark,functional_category,action_verbs
0,SWE-033,3.1,3.1.2,The project manager shall assess options for s...,"[Note: The assessment can include risk, cost, ...","[body, matrix]",3.1.2,The project manager shall assess options for s...,Center,5,X,,Software Life Cycle Planning,"[develop, assess]"
1,SWE-013,3.1,3.1.3,"The project manager shall develop, maintain, a...",[Note: The recommended practices and guideline...,"[body, matrix]",3.1.3,"The project manager shall develop, maintain, a...",Center,5,X,,Software Life Cycle Planning,"[develop, maintain]"
2,SWE-024,3.1,3.1.4,The project manager shall track the actual res...,[],"[body, matrix]",3.1.4,The project manager shall track the actual res...,Center,4,X,,Software Life Cycle Planning,[track]
3,SWE-034,3.1,3.1.5,The project manager shall define and document ...,[],"[body, matrix]",3.1.5,The project manager shall define and document ...,Center,4,X,,Software Life Cycle Planning,[document]
4,SWE-036,3.1,3.1.6,The project manager shall establish and mainta...,[Note: A list of typical software engineering ...,"[body, matrix]",3.1.6,The project manager shall establish and mainta...,Center,4,X,,Software Life Cycle Planning,"[develop, document, maintain, review]"
5,SWE-037,3.1,3.1.7,The project manager shall define and document ...,[],"[body, matrix]",3.1.7,The project manager shall define and document ...,Center,4,X,,Software Life Cycle Planning,"[develop, document, review]"
6,SWE-039,3.1,3.1.8,The project manager shall require the software...,[],"[body, matrix]",3.1.8,The project manager shall require the software...,Center,4,X,,Software Life Cycle Planning,"[develop, test, review]"
7,SWE-040,3.1,3.1.9,The project manager shall require the software...,[],"[body, matrix]",3.1.9,The project manager shall require the software...,Center,4,X,,Software Life Cycle Planning,"[develop, track]"
8,SWE-042,3.1,3.1.10,The project manager shall require the software...,[Note: The electronic access requirements for ...,"[body, matrix]",3.1.10,The project manager shall require the software...,Center,5,X,,Software Life Cycle Planning,[develop]
9,SWE-139,3.1,3.1.11,The project manager shall comply with the requ...,[],"[body, matrix]",3.1.11,The project manager shall comply with the requ...,Center,5,X,,Software Life Cycle Planning,[]


Cleaning extracted sections and ensuring proper extraction by checking:
1. A clause carrying 2+ SWE tags means the next clause's number was lost and it got swallowed
2. Body vs Appendix C matrix. Equal counts can hide a swap, so compare the sets.
3. Check for duplicates
4. Whole sections that produced zero clauses
5. Debris at the end of text or notes
6. Body clause number disagrees with the matrix's section number for the same SWE


In [54]:
EXPECTED_SECTIONS = [
    "3.1", "3.2", "3.3", "3.4", "3.5", "3.6", "3.7", "3.8", "3.9", "3.10", "3.11", "3.12",
    "4.1", "4.2", "4.3", "4.4", "4.5", "4.6",
    "5.1", "5.2", "5.3", "5.4", "5.5",
]

SWE_TAG = re.compile(r"\[(SWE-\d+)\]")

# Leftover fragments from headings that wrap onto a second line, plus Chapter 6 bleeding into 5.5.4
JUNK_TAIL = re.compile(
    r"(?:\n\s*\n\s*(?:&\s*Validation|Requirements)\s*$)"      # 2nd line of wrapped 3.6 / Ch.5 titles
    r"|(?:\n\s*(?:Content\s*\n\s*)?6\.1\s+Software Engineering Products[\s\S]*$)"  # Chapter 6 start
)

def section_key(s):
    return tuple(int(p) for p in str(s).split("."))

def audit_clause_list(df, matrix_dict):
    issues = {}

    # 1. A clause carrying 2+ SWE tags means the next clause's number was lost and it got swallowed
    multi = {}
    for _, row in df.iterrows():
        blob = row["text"] + "\n" + "\n".join(row["notes"])
        ids = list(dict.fromkeys(SWE_TAG.findall(blob)))
        if len(ids) > 1:
            multi[row["clause_id"]] = ids
    issues["clauses_with_multiple_swe_ids"] = multi

    # 2
    body_ids, matrix_ids = set(df["swe_id"]), set(matrix_dict)
    issues["in_matrix_not_in_body"] = sorted(matrix_ids - body_ids)
    issues["in_body_not_in_matrix"] = sorted(body_ids - matrix_ids)

    # 3
    issues["duplicate_swe_ids"] = sorted(df["swe_id"][df["swe_id"].duplicated()].unique())
    issues["duplicate_clause_ids"] = sorted(df["clause_id"][df["clause_id"].duplicated()].unique())

    # 4
    issues["missing_sections"] = [s for s in EXPECTED_SECTIONS if s not in set(df["section"])]

    # 5
    junk = []
    for _, row in df.iterrows():
        fields = [row["text"]] + list(row["notes"])
        if any(JUNK_TAIL.search(f) for f in fields):
            junk.append(row["clause_id"])

    issues["clauses_with_trailing_junk"] = junk

    # 6
    mismatch = {}
    for _, row in df.iterrows():
        m = matrix_dict.get(row["swe_id"])
        if m and m.get("matrix_section") and m["matrix_section"] != row["clause_id"]:
            mismatch[row["swe_id"]] = (row["clause_id"], m["matrix_section"])
    issues["clause_id_vs_matrix_section"] = mismatch

    return issues



In [55]:
def print_issues(issues):
    for name, val in issues.items():
        status = "OK" if not val else f"CHECK -> {val}"
        print(f"{name:32s} {status}")

qa_before = audit_clause_list(reqs_clauses_df, matrix_dict)
print_issues(qa_before)

clauses_with_multiple_swe_ids    CHECK -> {'5.1.9': ['SWE-045', 'SWE-086']}
in_matrix_not_in_body            CHECK -> ['SWE-086']
in_body_not_in_matrix            OK
duplicate_swe_ids                OK
duplicate_clause_ids             OK
missing_sections                 CHECK -> ['5.2']
clauses_with_trailing_junk       CHECK -> ['3.5.2', '3.12.1', '4.6.6', '5.5.4']
clause_id_vs_matrix_section      OK


Repairing the clause list: splitting merged clauses back into separate rows, removing leftover heading text, recomputing the annotations, and re-running the QA check

In [56]:
def split_multi_swe(row, matrix_dict):
    """Split one row whose text holds several [SWE-xxx] tags into one row per SWE ID."""
    text = row["text"]
    tags = list(SWE_TAG.finditer(text))
    if len(tags) <= 1:
        return [row.to_dict()]

    # Lettered sub-items (a., b., ...) after a tag still belong to it, so each swallowed
    # clause starts at the last blank line before its own tag
    bounds = [0]
    for this_tag, next_tag in zip(tags, tags[1:]):
        gap = text.rfind("\n\n", this_tag.end(), next_tag.start())
        bounds.append(gap if gap != -1 else this_tag.end())
    bounds.append(len(text))

    rows = []
    for i, tag in enumerate(tags):
        new = row.to_dict()
        swe_id = tag.group(1)
        new["swe_id"] = swe_id
        new["text"] = text[bounds[i]:bounds[i + 1]].strip()
        new["notes"] = list(row["notes"]) if i == len(tags) - 1 else []  # notes follow the last clause
        if i > 0:
            m = matrix_dict.get(swe_id, {})
            new.update(m)
            new["clause_id"] = m.get("matrix_section")
            new["section"] = ".".join(new["clause_id"].split(".")[:2]) if new["clause_id"] else None
            new["sources"] = ["body", "matrix"] if m else ["body"]
            if not m:
                print(f"WARNING: {swe_id} was split out but has no Appendix C row, so its clause_id is unknown")
        rows.append(new)
    return rows

def strip_junk(s):
    return JUNK_TAIL.sub("", s).rstrip()

repaired = []
for _, row in reqs_clauses_df.iterrows():
    repaired.extend(split_multi_swe(row, matrix_dict))

reqs_clauses_df = pd.DataFrame(repaired)
reqs_clauses_df["text"] = reqs_clauses_df["text"].apply(strip_junk)
reqs_clauses_df["notes"] = reqs_clauses_df["notes"].apply(lambda ns: [strip_junk(n) for n in ns if strip_junk(n)])

# recompute annotations so split-out rows get their own values, not the ones copied from the row they came from
reqs_clauses_df["functional_category"] = reqs_clauses_df["clause_id"].apply(apply_category)
reqs_clauses_df["action_verbs"] = reqs_clauses_df["text"].apply(extract_and_add_action_verbs)

reqs_clauses_df = (reqs_clauses_df
                   .sort_values("clause_id", key=lambda s: s.map(section_key))
                   .reset_index(drop=True))

qa_after = audit_clause_list(reqs_clauses_df, matrix_dict)
print(f"{len(reqs_clauses_df)} clauses after repair\n")
print_issues(qa_after)

100 clauses after repair

clauses_with_multiple_swe_ids    OK
in_matrix_not_in_body            OK
in_body_not_in_matrix            OK
duplicate_swe_ids                OK
duplicate_clause_ids             OK
missing_sections                 OK
clauses_with_trailing_junk       OK
clause_id_vs_matrix_section      OK


In [57]:
SRS_SCOPE = {
  "SWE-050": {"scope": "direct",  "evidence": "Requirements are uniquely identified and approved, including requirements for COTS/GOTS/MOTS/OSS or reused components."},
  "SWE-184": {"scope": "direct",  "evidence": "Safety constraints, controls, mitigations, and assumptions between hardware, operator, and software are stated."},
  "SWE-210": {"scope": "direct",  "evidence": "Requirements for collecting, reporting, and storing data on detected adversarial actions."},
  "SWE-134": {"scope": "partial",  "evidence": "Requirements covering items a-l: safe init, safe transitions, command checks, error handling, safe state.", "note": "Chat said partial: 'implement in the software' needs code/tests to prove."},
  "SWE-052": {"scope": "partial", "evidence": "Upward traceability to parent requirements and traceability to hazards. Design/code/test links live elsewhere."},
  "SWE-051": {"scope": "partial", "evidence": "Parent references on flowed-down requirements; derived requirements flagged with rationale."},
  "SWE-027": {"scope": "partial", "evidence": "Item a only: requirements the COTS/GOTS/MOTS/OSS/reused component must meet."},
  "SWE-157": {"scope": "partial", "evidence": "Access control and authentication requirements for software with communications capability."},
  "SWE-154": {"scope": "partial", "evidence": "Cybersecurity mitigations stated as security requirements."},
  "SWE-034": {"scope": "partial", "evidence": "Verification method and pass criteria per requirement.", "note": "Chat said direct."},
  "SWE-205": {"scope": "partial", "evidence": "Safety-critical requirements or components are flagged.", "note": "Chat said out: the determination is a separate SMA record."},
  "SWE-023": {"scope": "partial", "evidence": "NASA-STD-8739.8 safety-critical requirements expressed as software requirements."},
  "SWE-147": {"scope": "partial", "evidence": "Reusability requirements are specified.", "note": "Chat said direct."},
  "SWE-053": {"scope": "partial", "evidence": "Revision history, stable requirement IDs, and change markings."}
}

In [58]:
# with open("srs_scope.json") as f:
#     srs_scope = json.load(f)   # any SWE not listed is out of scope

srs_scope = SRS_SCOPE

unknown = set(srs_scope) - set(reqs_clauses_df["swe_id"])
assert not unknown, f"srs_scope.json lists SWE IDs not in the clause list: {unknown}"

reqs_clauses_df["srs_scope"] = reqs_clauses_df["swe_id"].map(lambda s: srs_scope.get(s, {}).get("scope", "out"))
reqs_clauses_df["srs_evidence"] = reqs_clauses_df["swe_id"].map(lambda s: srs_scope.get(s, {}).get("evidence"))
srs_clauses_df = reqs_clauses_df[reqs_clauses_df["srs_scope"] != "out"].reset_index(drop=True)

print(reqs_clauses_df["srs_scope"].value_counts().to_string())
srs_clauses_df[["swe_id", "clause_id", "functional_category", "srs_scope", "srs_evidence"]]

srs_scope
out        86
partial    11
direct      3


,swe_id,clause_id,functional_category,srs_scope,srs_evidence
0,SWE-034,3.1.5,Software Life Cycle Planning,partial,Verification method and pass criteria per requ...
1,SWE-027,3.1.14,Software Life Cycle Planning,partial,Item a only: requirements the COTS/GOTS/MOTS/O...
2,SWE-205,3.7.1,Safety-Critical Software,partial,Safety-critical requirements or components are...
3,SWE-023,3.7.2,Safety-Critical Software,partial,NASA-STD-8739.8 safety-critical requirements e...
4,SWE-134,3.7.3,Safety-Critical Software,partial,"Requirements covering items a-l: safe init, sa..."
5,SWE-147,3.10.1,Software Reuse,partial,Reusability requirements are specified.
6,SWE-154,3.11.3,Software Cybersecurity,partial,Cybersecurity mitigations stated as security r...
7,SWE-157,3.11.4,Software Cybersecurity,partial,Access control and authentication requirements...
8,SWE-210,3.11.8,Software Cybersecurity,direct,"Requirements for collecting, reporting, and st..."
9,SWE-052,3.12.1,Software Bi-Directional Traceability,partial,Upward traceability to parent requirements and...


In [59]:
reqs_clauses_df.to_csv("annotated_nasa_clauses.csv", index=False)
srs_clauses_df.to_csv("srs_addressable_clauses.csv", index=False)
print(f"Saved {len(reqs_clauses_df)} annotated clauses and {len(srs_clauses_df)} SRS-addressable clauses")

Saved 100 annotated clauses and 14 SRS-addressable clauses


In [60]:
from google.colab import files
uploaded = files.upload()

Saving Starhawk Mission Computer SRS.md to Starhawk Mission Computer SRS (3).md


In [61]:
filename = next(iter(uploaded))
srs_text = uploaded[filename].decode("utf-8-sig")

print("Loaded:", filename)
print(srs_text[:500])

Loaded: Starhawk Mission Computer SRS (3).md
# Software Requirements Specification (SRS)
## Starhawk Space Fighter Mission Computer

**Document ID:** SH-SRS-001  
**Version:** 1.0  
**Status:** Draft for Review  

---

# 1. Introduction

## 1.1 Purpose

This Software Requirements Specification defines the software requirements for the **Starhawk Space Fighter Mission Computer (SFMC)**.

The SFMC is the primary onboard computer for the Starhawk single-seat space fighter. The software assists the pilot with spacecraft navigation, propulsion 


In [62]:
import re
import pandas as pd

pattern = (
    r"^\*\*(SFMC-REQ-\d{3})\s*—\s*(.*?)\*\*\s*\n"
    r"(.*?)(?=^\*\*SFMC-REQ-|^#|\Z)"
)

rows = []

for req_id, title, text in re.findall(
    pattern, srs_text, flags=re.MULTILINE | re.DOTALL
):
    text = re.sub(r"^---\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\s+", " ", text).strip()

    rows.append({
        "requirement_id": req_id,
        "title": title,
        "text": text
    })

srs_df = pd.DataFrame(rows)

print(f"Found {len(srs_df)} requirements")
display(srs_df.head())

Found 54 requirements


,requirement_id,title,text
0,SFMC-REQ-001,Position Display,The SFMC shall display the spacecraft's curren...
1,SFMC-REQ-002,Navigation Update,The SFMC shall update the displayed spacecraft...
2,SFMC-REQ-003,Destination Selection,The SFMC shall allow the pilot to select a nav...
3,SFMC-REQ-004,Route Calculation,"When the pilot selects a destination, the SFMC..."
4,SFMC-REQ-005,Autopilot,"When Autopilot mode is enabled, the SFMC shall..."


In [63]:
# Phrases that may need clarification.
# These rules flag possible issues for human review.
rules = {
    "quickly": "Specify a measurable response-time limit.",
    "immediately": "Clarify the maximum allowed response time.",
    "without noticeable delay": "Response time is subjective.",
    "reasonably close": "Define an acceptable arrival distance.",
    "extremely accurate": "Define the accuracy metric and limit.",
    "important": "Identify exactly which data or communications are covered.",
    "appropriate corrective action": "Define the required corrective action.",
    "whenever possible": "Define when continued operation is required.",
    "seriously damaged": "Define the condition that triggers this behavior.",
    "sufficient storage": "Define storage capacity or the expected workload.",
    "state-of-the-art": "Identify the applicable security criteria.",
    "securely encrypted": "Identify the applicable encryption requirements.",
    "safe and efficient": "Define safety and efficiency criteria.",
    "safest destination": "Define destination selection criteria.",
    "safe location": "Define what qualifies as a safe location.",
    "friendly spacecraft": "Define the criteria for identifying friendly, hostile, or unknown spacecraft.",
    "greatest danger": "Define the threat-priority factors and ranking method.",
    "nonessential spacecraft functions": "Identify the functions that may be delayed or disabled in Emergency mode."
}

def check_requirement(text):
    matches = []

    for phrase, explanation in rules.items():
        if re.search(r"\b" + re.escape(phrase) + r"\b", text, re.I):
            matches.append(f"{phrase}: {explanation}")

    return matches

baseline_df = srs_df.copy()

baseline_df["findings"] = baseline_df["text"].apply(check_requirement)

baseline_df["status"] = baseline_df["findings"].apply(
    lambda findings: "NEEDS_REVIEW" if findings else "NO_RULE_MATCH"
)

print(baseline_df["status"].value_counts())

display(
    baseline_df.loc[
        baseline_df["status"] == "NEEDS_REVIEW",
        ["requirement_id", "findings"]
    ]
)

status
NO_RULE_MATCH    37
NEEDS_REVIEW     17
Name: count, dtype: int64


,requirement_id,findings
3,SFMC-REQ-004,[safe and efficient: Define safety and efficie...
5,SFMC-REQ-006,[reasonably close: Define an acceptable arriva...
9,SFMC-REQ-010,[appropriate corrective action: Define the req...
14,SFMC-REQ-015,[friendly spacecraft: Define the criteria for ...
15,SFMC-REQ-016,[extremely accurate: Define the accuracy metri...
17,SFMC-REQ-018,[greatest danger: Define the threat-priority f...
20,SFMC-REQ-021,[quickly: Specify a measurable response-time l...
24,SFMC-REQ-025,[important: Identify exactly which data or com...
29,SFMC-REQ-030,[immediately: Clarify the maximum allowed resp...
31,SFMC-REQ-032,[nonessential spacecraft functions: Identify t...


In [64]:
# Convert findings into readable text for the CSV
export_df = baseline_df.copy()
export_df["findings"] = export_df["findings"].apply(
    lambda items: " | ".join(items)
)

export_df.to_csv("baseline_results.csv", index=False)

from google.colab import files
files.download("baseline_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>